In [39]:
# pip install pandas numpy matplotlib openpyxl

## DATASET OVERVIEW

In [40]:
import pandas as pd
import matplotlib.pyplot as plt

_df_train= pd.read_csv("algebra_2006_2007_train.txt", sep="\t")
_df_test = pd.read_csv("algebra_2006_2007_test.txt", sep="\t")

In [41]:
# Basic info initial dataset
print("Train shape:",_df_train.shape)
print("Columns:", _df_train.columns.to_list())
print("Number of students:", _df_train['Anon Student Id'].nunique())
print("Number of problems:", _df_train['Problem Name'].nunique())
print("Number of steps:", _df_train['Step Name'].nunique())
print("Number of KCs:", _df_train['KC(Default)'].nunique())


Train shape: (2270384, 19)
Columns: ['Row', 'Anon Student Id', 'Problem Hierarchy', 'Problem Name', 'Problem View', 'Step Name', 'Step Start Time', 'First Transaction Time', 'Correct Transaction Time', 'Step End Time', 'Step Duration (sec)', 'Correct Step Duration (sec)', 'Error Step Duration (sec)', 'Correct First Attempt', 'Incorrects', 'Hints', 'Corrects', 'KC(Default)', 'Opportunity(Default)']
Number of students: 1338
Number of problems: 91913
Number of steps: 316896
Number of KCs: 1703


## KC election

Some steps have multiple KCs associated with them; when this occurs, they are separated by ~~. In this section, we create one row per KC and then classify them to obtain a reduced and interpretable set of KCs related to first-degree linear equations:

1. Remove Constant – Moving constant terms from one side of the equation to the other.
2. Combine Like Terms – Simplifying expressions by combining terms of the same type.
3. Consolidate Variable Terms – Bringing all variable terms to one side of the equation.
4. Eliminate Parentheses / Distribute – Removing parentheses by applying the distributive property.
5. Multiply or Divide Terms – Performing multiplication or division to simplify expressions.
6. Remove Coefficient – Eliminating the coefficient of the variable to isolate it.
7. Handle Negative Variable – Correctly managing negative signs when isolating the variable.

In [ ]:
def map_kc(kc):
    if pd.isna(kc):
        return None
    
    if "SkillRule: Remove coefficient;" in kc:
        return "Remove Coefficient"
    elif "SkillRule: Remove positive coefficient;" in kc:
        return "Remove Coefficient"
    elif "SkillRule: Remove negative coefficient;" in kc:
        return "Remove Coefficient"
    elif "SkillRule: Multiply/Divide; [Typein]" in kc:
        return "Remove Coefficient"
    
    elif "SkillRule: Remove constant;" in kc:
        return "Move Constants Across the Equation"
    elif "SkillRule: Add/Subtract;" in kc:
        return "Move Constants Across the Equation"
    elif "SkillRule: ax+b=c, negative;" in kc:
        return "Move Constants Across the Equation"
    elif "move constant" in kc:
        return "Move Constants Across the Equation"
    elif "[SolverOperation add]" in kc or "[SolverOperation subtract]" in kc:
        return "Move Constants Across the Equation"
    elif "SkillRule: Isolate positive;" in kc:
        return "Move Constants Across the Equation"
    elif "SkillRule: Isolate negative;" in kc:
        return "Move Constants Across the Equation"

    elif "SkillRule: Consolidate vars, no coeff;" in kc:
        return "Move Variable Terms to One Side"
    elif "SkillRule: Consolidate vars with coeff;" in kc:
        return "Move Variable Terms to One Side"
    elif "SkillRule: Consolidate vars, any;" in kc:
        return "Move Variable Terms to One Side"
    elif "SkillRule: Extract to consolidate vars;" in kc:
        return "Move Variable Terms to One Side"
    
    elif "SkillRule: Eliminate Parens;" in kc:
        return "Expand / Eliminate Parentheses"
    elif "SkillRule: Select Eliminate Parens;" in kc:
        return "Expand / Eliminate Parentheses"
    elif "SkillRule: Calculate Eliminate Parens;" in kc:
        return "Expand / Eliminate Parentheses"
    elif "SkillRule: Do Eliminate Parens - whole;" in kc:
        return "Expand / Eliminate Parentheses"
    elif "distribute-sp" in kc.lower():
        return "Expand / Eliminate Parentheses"
    elif "CLT nested, parens (null" in kc:
        return "Expand / Eliminate Parentheses"    
    elif "perform-mult-whole-sp" in kc:
        return "Expand / Eliminate Parentheses"

    elif "SkillRule: Make variable positive;" in kc:
        return "Normalize Negative Variable / Sign"
    elif "SkillRule: Calculate negative coefficient;" in kc:
        return "Normalize Negative Variable / Sign"
    
    elif "SkillRule: Combine like terms, no var;" in kc:
        return "Combine Like Terms"
    elif "SkillRule: Select Combine Terms;" in kc:
        return "Combine Like Terms"
    elif "SkillRule: Do Combine Terms - Whole;" in kc:
        return "Combine Like Terms"
    elif "combineLikeTerms" in kc or "CLT" in kc:
        return "Combine Like Terms"
    elif "combine-like-terms" in kc.lower():
        return "Combine Like Terms"

    return "Other"


def preprocess_kc(
    df,
    remove_other=False,
    drop_original_kc_cols=False,
    filter_irrelevant=False
):
    df = df.copy()

    # Remove rows without KC(Default)
    df = df.dropna(subset=["KC(Default)"])

    # Split multiple KCs
    df["KC"] = df["KC(Default)"].astype(str).str.split("~~")

    # Explode to one row per KC
    df = df.explode("KC")

    # Clean KC
    df["KC"] = df["KC"].astype(str).str.strip()
    df = df[df["KC"] != ""]
    df = df[df["KC"].str.lower() != "nan"]

    # Optional filter for clearly irrelevant domains
    if filter_irrelevant:
        irrelevant_patterns = [
            "action", "compare", "axis", "-row",
            "unspecified", "unknown", "any form", "entering a",
            "plot", "graph", "probability", "median", "mean",
            "mode", "rate", "ratio", "scientific notation","slope",
            "square root", "hypotenuse", "geometry", "quadrat", "exponent"
        ]

        def is_relevant(kc):
            if pd.isna(kc):
                return False
            kc = kc.lower()
            return not any(pattern in kc for pattern in irrelevant_patterns)

        df = df[df["KC"].apply(is_relevant)]

    df["new_KC"] = df["KC"].apply(map_kc)
    if remove_other:
        df = df[df["new_KC"] != "Other"]
    # Optional drop of original KC columns
    if drop_original_kc_cols:
        cols_to_drop = [col for col in ["KC(Default)", "KC"] if col in df.columns]
        df = df.drop(columns=cols_to_drop)

    # Clean index
    df = df.reset_index(drop=True)

    return df

In [43]:
# 1. KEEP ONLY RELEVANT COLUMNS
cols_needed = [
    "KC(Default)",
    "Problem Name",
    "Step Name",
    "Correct First Attempt",
    "Anon Student Id"
]

df = _df_train[cols_needed].copy()

df = preprocess_kc(df, remove_other=True)

# 4. BUILD MAIN KC -> PROBLEM -> STEP TABLE
kc_problem_step = (
    df.groupby(["KC", "new_KC", "Problem Name", "Step Name"])
      .agg(
          appearances=("KC", "size"),
          students=("Anon Student Id", "nunique"),
          avg_correct=("Correct First Attempt", "mean")
      )
      .reset_index()
      .sort_values(["new_KC", "appearances"], ascending=[True, False])
)

# 5. UNIQUE KCs WITH COUNTS
kc_summary = (
    df.groupby("KC")
      .agg(
          total_appearances=("KC", "size"),
          n_problems=("Problem Name", "nunique"),
          n_steps=("Step Name", "nunique"),
          n_students=("Anon Student Id", "nunique"),
          avg_correct=("Correct First Attempt", "mean")
      )
      .reset_index()
      .sort_values("total_appearances", ascending=False)
)

# 6. EXPORT FOR MANUAL REVIEW
kc_problem_step.to_excel("kc_problem_step_mapping.xlsx", index=False)
kc_summary.to_excel("kc_summary.xlsx", index=False)


Final dataset with only relevant KC columns

In [44]:
df_train = preprocess_kc(_df_train, remove_other=True, drop_original_kc_cols=True, filter_irrelevant=True)
df_test = preprocess_kc(_df_test, remove_other=True, drop_original_kc_cols=True, filter_irrelevant=True)

# Basic info
print("Train shape:", df_train.shape)
print("Columns:", df_train.columns.to_list())
print("Number of students:", df_train['Anon Student Id'].nunique())
print("Number of problems:", df_train['Problem Name'].nunique())
print("Number of steps:", df_train['Step Name'].nunique())
print("Number of KCs:", df_train['new_KC'].nunique())

print(df_train["new_KC"].value_counts())

Train shape: (371497, 19)
Columns: ['Row', 'Anon Student Id', 'Problem Hierarchy', 'Problem Name', 'Problem View', 'Step Name', 'Step Start Time', 'First Transaction Time', 'Correct Transaction Time', 'Step End Time', 'Step Duration (sec)', 'Correct Step Duration (sec)', 'Error Step Duration (sec)', 'Correct First Attempt', 'Incorrects', 'Hints', 'Corrects', 'Opportunity(Default)', 'new_KC']
Number of students: 1165
Number of problems: 79821
Number of steps: 207052
Number of KCs: 6
new_KC
Move Constants Across the Equation    105059
Move Variable Terms to One Side        80320
Remove Coefficient                     80289
Expand / Eliminate Parentheses         59619
Combine Like Terms                     45275
Normalize Negative Variable / Sign       935
Name: count, dtype: int64


## Remove duplicate entries

In [45]:
subset_cols = [
    "Anon Student Id",
    "Problem Name",
    "Step Name",
    "new_KC",
    "Correct First Attempt"
]

# Sort first
df_train = df_train.sort_values(
    by=["Anon Student Id", "Problem Name", "Step Name"]
).reset_index(drop=True)

df_test = df_test.sort_values(
    by=["Anon Student Id", "Problem Name", "Step Name"]
).reset_index(drop=True)

# THEN compute duplicates
duplicates = df_train.duplicated(subset=subset_cols)

n_duplicates = duplicates.sum()
percentage = (n_duplicates / len(df_train)) * 100

print(f"Duplicated interactions: {n_duplicates}")
print(f"Percentage: {percentage:.2f}%")

# Now this works correctly
df_train = df_train.drop_duplicates(subset=subset_cols)
df_test = df_test.drop_duplicates(subset=subset_cols)

duplicates_after = df_train.duplicated(subset=subset_cols).sum()
print(f"Remaining duplicates: {duplicates_after}")




Duplicated interactions: 3231
Percentage: 0.87%
Remaining duplicates: 0


Notas: 
- cata studiante tiene su P(L) prob. of learning 
- este dataset nos ayuda a obtener las probabilidades Pl0, pt, pg, ps para cada KC 
- despues de cada ejercicio realizadp la P(L) del estudiante se actualiza

# mirar si cambiar a 6 kc!!! 
cambiar consolidate? 
